# Modeloplossing — Checkpoint Dimensie reductie

Dit is de **modeloplossing** van het checkpoint. De bestandsnaam mag niet gewijzigd worden: de checks in `checks/` verwijzen naar dit notebook (via `CHECKPOINT_SOLUTIONS=1` wordt het als referentie gebruikt).

De opgave-versie (`week07/dim_reductie_checkpoint_opgave.ipynb`) heeft dezelfde celstructuur met TODO's die de student invult. Gebruik voor elke dataset `PCA(n_components=0.95)` op de gestandaardiseerde data en bereken de datareductie als `(1 - n_components / oorspronkelijk_aantal_features) * 100`.


## Oefening 1 — SECOM

De SECOM-dataset bevat 590 sensormetingen van halfgeleiderproductie. Het bestand bevat veel ontbrekende waarden; vul ze in met het kolomgemiddelde.

Pas PCA toe zodat **95% van de variantie behouden blijft**. Bewaar het aantal componenten in `n_components_secom`, de gereduceerde data in `X_pca_secom` en de procentuele datareductie in `reduction_pct_secom`.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Pad naar bestanden
data_path = "../data/secom/secom.data"
labels_path = "../data/secom/secom_labels.data"

# Dataset laden (spatie is de separator in SECOM)
X = pd.read_csv(data_path, sep="\\s+", header=None)
y = pd.read_csv(labels_path, sep="\\s+", header=None, usecols=[0])  # labels in eerste kolom

# Ontbrekende waarden invullen met het kolomgemiddelde
X = X.fillna(X.mean())

# Data standaardiseren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
pca = PCA(n_components=0.95)
X_pca_secom = pca.fit_transform(X_scaled)

n_components_secom = X_pca_secom.shape[1]
reduction_pct_secom = (1 - n_components_secom / X.shape[1]) * 100


In [ ]:
pca_2 = PCA(n_components=2)
X_2 = pca_2.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
plt.scatter(X_2[:, 0], X_2[:, 1], c=y[0], cmap="coolwarm", s=20)
plt.title("SECOM PCA (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

pca_3 = PCA(n_components=3)
X_3 = pca_3.fit_transform(X_scaled)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_3[:, 0], X_3[:, 1], X_3[:, 2], c=y[0], s=20)
ax.set_title("SECOM PCA (3D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()


## Oefening 2 — Steel Plates Faults

De Steel Plates Faults-dataset bevat 27 metingen van staalplaten. De kolomnamen staan in `data/steel_plates/Faults27x7_var`; de laatste 7 kolommen zijn binaire foutlabels.

Pas PCA toe zodat **95% van de variantie behouden blijft** op de 27 meetkolommen. Bewaar het aantal componenten in `n_components_steel`, de gereduceerde data in `X_pca_steel` en de procentuele datareductie in `reduction_pct_steel`.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Pad naar bestanden
data_path = "../data/steel_plates/Faults.NNA"
names_path = "../data/steel_plates/Faults27x7_var"

# Kolomnamen laden
with open(names_path, "r") as f:
    colnames = [line.strip() for line in f.readlines()]

# Data laden
df = pd.read_csv(data_path, sep="\\s+", header=None, names=colnames)

# Eerste 27 kolommen zijn de metingen, de laatste 7 zijn binaire foutlabels
X = df.iloc[:, :27]
y = df.iloc[:, 27:]

# Ontbrekende waarden invullen met het kolomgemiddelde
X = X.fillna(X.mean())

# Data standaardiseren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
pca = PCA(n_components=0.95)
X_pca_steel = pca.fit_transform(X_scaled)

n_components_steel = X_pca_steel.shape[1]
reduction_pct_steel = (1 - n_components_steel / X.shape[1]) * 100


In [ ]:
y_combined = y.idxmax(axis=1)  # kolomnaam met de hoogste waarde = fouttype
y_combined[y.sum(axis=1) == 0] = "No_Fault"

pca_2 = PCA(n_components=2)
X_2 = pca_2.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
for label in y_combined.unique():
    mask = y_combined == label
    plt.scatter(X_2[mask, 0], X_2[mask, 1], s=20, label=label)
plt.title("Steel Plates PCA (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.show()

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

pca_3 = PCA(n_components=3)
X_3 = pca_3.fit_transform(X_scaled)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_3[:, 0], X_3[:, 1], X_3[:, 2], c=y_combined.astype("category").cat.codes, s=20)
ax.set_title("Steel Plates PCA (3D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()


## Oefening 3 — APS Failure at Scania Trucks

De APS-dataset bevat 170 sensormetingen van vrachtwagens. De eerste 20 regels van de CSV-bestanden zijn metadata; de testset bevat geen labels.

Combineer train en test, vul ontbrekende waarden in en standaardiseer. Pas PCA toe zodat **95% van de variantie behouden blijft**. Bewaar het aantal componenten in `n_components_aps`, de gereduceerde data in `X_pca_aps` en de procentuele datareductie in `reduction_pct_aps`.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Train- en testdata laden (eerste 20 regels zijn metadata)
train = pd.read_csv("../data/aps_failure/aps_failure_training_set.csv", skiprows=20, na_values="na")
test = pd.read_csv("../data/aps_failure/aps_failure_test_set.csv", skiprows=20, na_values="na")

X_train = train.drop(columns=["class"])
y_train = train["class"]
X_test = test.drop(columns=["class"])

# Combineer train + test voor de PCA (unsupervised setup)
X_all = pd.concat([X_train, X_test], ignore_index=True)

# Ontbrekende waarden invullen met het kolomgemiddelde
X_all = X_all.fillna(X_all.mean())

# Data standaardiseren
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)


In [ ]:
pca = PCA(n_components=0.95)
X_pca_aps = pca.fit_transform(X_scaled)

n_components_aps = X_pca_aps.shape[1]
reduction_pct_aps = (1 - n_components_aps / X_all.shape[1]) * 100


In [ ]:
pca_2 = PCA(n_components=2)
X_2 = pca_2.fit_transform(X_scaled[:len(y_train)])

plt.figure(figsize=(8, 6))
plt.scatter(X_2[:, 0], X_2[:, 1], c=(y_train == "pos").astype(int), s=20, alpha=0.8)
plt.title("APS Failures PCA (2D)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

pca_3 = PCA(n_components=3)
X_3 = pca_3.fit_transform(X_scaled[:len(y_train)])

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_3[:, 0], X_3[:, 1], X_3[:, 2], c=(y_train == "pos").astype(int), s=20, alpha=0.8)
ax.set_title("APS Failures PCA (3D)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()


## Meerkeuzevragen

### Vraag 1 — Standaardiseren
Waarom moet je de features **standaardiseren** vóór je PCA toepast?

- **A)** Omdat PCA alleen werkt op gehele getallen.
- **B)** Omdat features op verschillende schalen anders onevenredig veel gewicht krijgen en de PCA dan vooral de grote schalen weerspiegelt.
- **C)** Omdat standaardiseren het aantal features vermindert.
- **D)** Omdat standaardiseren automatisch 95% van de variantie behoudt.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_scaling`.

### Vraag 2 — n_components=0.95
Wat betekent `PCA(n_components=0.95)`?

- **A)** Het model houdt 95 van de oorspronkelijke features over.
- **B)** Het model houdt 95% van de datapunten over.
- **C)** Het model kiest het kleinste aantal componenten dat samen minstens 95% van de variantie verklaart.
- **D)** Het model kiest altijd exact 95 componenten.

Bewaar je antwoord (`"A"`, `"B"`, `"C"` of `"D"`) in `antwoord_95`.


In [ ]:
# Vraag 1: standaardiseren zorgt dat alle features gelijk gewicht krijgen.
antwoord_scaling = "B"

# Vraag 2: n_components=0.95 kiest het kleinste aantal componenten dat
# samen minstens 95% van de variantie verklaart.
antwoord_95 = "C"
